# Poglavlje 7: Izgradnja chat aplikacija
## Microsoft Foundry Models API Brzi početak

Ovaj bilježnik je prilagođen iz [Azure OpenAI uzoraka na repozitoriju](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) koji uključuje bilježnice koje pristupaju [Azure OpenAI](notebook-azure-openai.ipynb) uslugama.

> **Napomena:** GitHub modeli se povlače krajem srpnja 2026. Ovaj bilježnik sada cilja na [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), koji nudi isti katalog modela besplatan za isprobavanje i iskustvo Azure AI Inference SDK-a.


# Pregled  
"Veliki jezični modeli su funkcije koje preslikavaju tekst u tekst. Za zadani ulazni tekst, veliki jezični model pokušava predvidjeti sljedeći tekst koji će doći"(1). Ovaj "brzi početak" bilježnica će korisnicima predstaviti osnovne LLM koncepte, ključne zahtjeve paketa za početak rada s AML-om, nježni uvod u dizajn promptova i nekoliko kratkih primjera različitih slučajeva upotrebe. 


## Sadržaj  

[Pregled](#overview)  
[Kako koristiti OpenAI uslugu](#how-to-use-openai-service)  
[1. Kreiranje vaše OpenAI usluge](#1.-creating-your-openai-service)  
[2. Instalacija](#2.-installation)    
[3. Podaci za prijavu](#3.-credentials)  

[Primjeri upotrebe](#use-cases)    
[1. Sažeti tekst](#1.-summarize-text)  
[2. Klasificirati tekst](#2.-classify-text)  
[3. Generirati nove nazive proizvoda](#3.-generate-new-product-names)  
[4. Dodatno podešavanje klasifikatora](#4.fine-tune-a-classifier)  

[Reference](#references)


### Izgradite svoj prvi prompt  
Ova kratka vježba pružit će osnovni uvod za slanje promptova modelu u Microsoft Foundry Models za jednostavan zadatak "sažimanje".


**Koraci**:  
1. Instalirajte `azure-ai-inference` biblioteku u vašem Python okruženju, ako to već niste učinili.  
2. Učitajte standardne pomoćne biblioteke i postavite svoje Microsoft Foundry Models vjerodajnice.  
3. Odaberite model za svoj zadatak  
4. Kreirajte jednostavan prompt za model  
5. Pošaljite svoj zahtjev model API-ju!


### 1. Instalirajte `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Uvoz bibliotek pomoćnika i instanciranje vjerodajnica


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Pronalaženje pravog modela  
Modeli poput GPT-4o i GPT-4o mini mogu razumjeti i generirati prirodni jezik, a dostupni su u Microsoft Foundry Models katalogu zajedno s modelima iz Meta, Mistral, Cohere i Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Dizajn upita  

"Čarolija velikih jezičnih modela je u tome što, trenirajući se na minimiziranju ove pogreške predviđanja kroz ogromne količine teksta, modeli na kraju uče koncepte korisne za ta predviđanja. Na primjer, oni uče koncepte poput"(1):

* kako se pravilno piše
* kako funkcionira gramatika
* kako parafrazirati
* kako odgovarati na pitanja
* kako voditi razgovor
* kako pisati na mnogim jezicima
* kako kodirati
* itd.

#### Kako kontrolirati veliki jezični model  
"Od svih ulaza u veliki jezični model, najutjecajniji je tekstualni upit(1).

Velike jezične modele može se potaknuti na generiranje izlaza na nekoliko načina:

Uputa: Recite modelu što želite
Dovršenje: Potaknite model da dovrši početak onoga što želite
Demonstracija: Pokažite modelu što želite, s:
Nekoliko primjera u upitu
Stotinama ili tisućama primjera u skupu za fino podešavanje"



#### Postoje tri osnovna pravila za stvaranje upita:

**Prikaži i reci**. Jasno naznačite što želite, bilo kroz upute, primjere ili kombinaciju oboje. Ako želite da model rangira listu stavki po abecedi ili klasificira odlomak prema sentimentu, pokažite mu da je to ono što tražite.

**Osigurajte kvalitetne podatke**. Ako pokušavate izraditi klasifikator ili natjerati model da slijedi obrazac, pobrinite se da imate dovoljno primjera. Obavezno pročitajte svoje primjere – model je obično dovoljno pametan da primijeti osnovne pravopisne greške i da vam odgovori, ali također može pretpostaviti da je to namjerno i to može utjecati na odgovor.

**Provjerite svoje postavke.** Postavke temperature i top_p kontroliraju koliko je model determinističan u generiranju odgovora. Ako tražite odgovor gdje postoji samo jedan ispravan odgovor, želite te vrijednosti postaviti niže. Ako tražite raznovrsnije odgovore, možda ćete ih htjeti postaviti više. Najveća pogreška koju ljudi rade s ovim postavkama je pretpostavka da su one kontrole "pametnosti" ili "kreativnosti".


Izvor: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Predaj!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Ponovite isti poziv, kako se rezultati uspoređuju?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Sažmi tekst  
#### Izazov  
Sažmi tekst dodajući 'tl;dr:' na kraj teksta. Primijeti kako model razumije kako izvršiti niz zadataka bez dodatnih uputa. Možeš eksperimentirati s opisnijim uputama od tl;dr kako bi promijenio ponašanje modela i prilagodio primljeni sažetak(3).  

Nedavni rad pokazao je značajna poboljšanja na mnogim NLP zadacima i standardima pred-treniniranjem na velikom korpusu teksta, a zatim dodatnim podešavanjem na određenom zadatku. Iako je arhitektura obično neovisna o zadatku, ova metoda još uvijek zahtijeva skupove podataka za fino podešavanje specifične za zadatak, s tisućama ili desetkama tisuća primjera. Za razliku od toga, ljudi općenito mogu izvesti novi jezični zadatak iz samo nekoliko primjera ili jednostavnih uputa - nešto s čime se trenutačni NLP sustavi još uvijek uglavnom muče. Ovdje pokazujemo da povećanje veličine jezičnih modela značajno poboljšava izvedbu na neovisnim zadacima s malo primjera, ponekad čak dostižući konkurentnost s dosadašnjim vrhunskim metodama finog podešavanja.  



Tl;dr


# Vježbe za nekoliko slučajeva upotrebe  
1. Sažmi tekst  
2. Klasificiraj tekst  
3. Generiraj nova imena proizvoda


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Klasificirajte tekst  
#### Izazov  
Klasificirajte stavke u kategorije dana u trenutku izvođenja. U sljedećem primjeru dajemo i kategorije i tekst za klasifikaciju u upitu(*playground_reference). 

Upit korisnika: Pozdrav, jedan od tipki na mom laptop tipkovnici se nedavno pokvario i trebat će mi zamjena:

Klasificirana kategorija:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Generirajte nove nazive proizvoda
#### Izazov
Izradite nazive proizvoda na temelju primjera riječi. Ovdje u upitu uključujemo informacije o proizvodu za koji ćemo generirati nazive. Također pružamo sličan primjer kako bismo pokazali željeni uzorak. Postavili smo i visoku vrijednost temperature kako bismo povećali nasumičnost i kreativnost odgovora.

Opis proizvoda: Kućni uređaj za izradu milkshakea
Početne riječi: brz, zdrav, kompaktan.
Nazivi proizvoda: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Opis proizvoda: Par cipela koji može stati na bilo koju veličinu stopala.
Početne riječi: prilagodljiv, odgovarajući, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Reference  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Najbolje prakse za fino podešavanje GPT modela za klasifikaciju teksta](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Za dodatnu pomoć  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Suradnici
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
